<a href="https://colab.research.google.com/github/Nazihbenbrahim/-SentinelAI-/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — import libraries

import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [2]:
# Cell 2 — get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"


--2025-12-06 22:39:23--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 104.26.3.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358233 (350K) [text/tab-separated-values]
Saving to: ‘train-data.tsv’

train-data.tsv      100%[===================>] 349.84K  --.-KB/s    in 0.03s   

2025-12-06 22:39:23 (10.3 MB/s) - ‘train-data.tsv’ saved [358233/358233]

--2025-12-06 22:39:23--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 104.26.3.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 118774 (116K) [text/tab-separated-values]
Saving to: ‘valid-data.tsv’

valid-data.tsv      100%[==============

In [3]:
# Cell 3 — Charger les données et préparer les séquences de texte

# Charger les fichiers TSV : colonne 0 = label, colonne 1 = message
train_df = pd.read_csv(
    train_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

test_df = pd.read_csv(
    test_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

# Mapper les labels : ham -> 0, spam -> 1
label_map = {'ham': 0, 'spam': 1}
train_df['label_num'] = train_df['label'].map(label_map)
test_df['label_num'] = test_df['label'].map(label_map)

# Features et labels
x_train = train_df['message'].values
y_train = train_df['label_num'].values

x_test = test_df['message'].values
y_test = test_df['label_num'].values

# Tokenisation et padding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(x_train)

max_length = 100
trunc_type = 'post'
padding_type = 'post'

train_sequences = tokenizer.texts_to_sequences(x_train)
train_padded = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type
)

test_sequences = tokenizer.texts_to_sequences(x_test)
test_padded = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type
)


In [4]:
# Cell 4 — Construire et entraîner le modèle de classification spam/ham

embedding_dim = 16

model = keras.Sequential([
    keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(24, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')  # probabilité de spam
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    train_padded,
    y_train,
    epochs=10,          # tu peux monter à 15–20 si tu veux
    batch_size=32,
    validation_split=0.2,
    verbose=2
)

# Optionnel : évaluer sur le jeu de test
test_loss, test_acc = model.evaluate(test_padded, y_test, verbose=0)
print("Test accuracy:", test_acc)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
105/105 - 3s - 25ms/step - accuracy: 0.8600 - loss: 0.3992 - val_accuracy: 0.8612 - val_loss: 0.3747
Epoch 2/10
105/105 - 1s - 6ms/step - accuracy: 0.8672 - loss: 0.3621 - val_accuracy: 0.8612 - val_loss: 0.3687
Epoch 3/10
105/105 - 1s - 14ms/step - accuracy: 0.8672 - loss: 0.3563 - val_accuracy: 0.8612 - val_loss: 0.3611
Epoch 4/10
105/105 - 1s - 9ms/step - accuracy: 0.8672 - loss: 0.3415 - val_accuracy: 0.8612 - val_loss: 0.3364
Epoch 5/10
105/105 - 1s - 7ms/step - accuracy: 0.8672 - loss: 0.3011 - val_accuracy: 0.8612 - val_loss: 0.2745
Epoch 6/10
105/105 - 1s - 5ms/step - accuracy: 0.8902 - loss: 0.2235 - val_accuracy: 0.9079 - val_loss: 0.1929
Epoch 7/10
105/105 - 1s - 5ms/step - accuracy: 0.9515 - loss: 0.1477 - val_accuracy: 0.9665 - val_loss: 0.1306
Epoch 8/10
105/105 - 1s - 7ms/step - accuracy: 0.9716 - loss: 0.1006 - val_accuracy: 0.9749 - val_loss: 0.0974
Epoch 9/10
105/105 - 1s - 10ms/step - accuracy: 0.9791 - loss: 0.0767 - val_accuracy: 0.9809 - val_loss: 0.079

In [5]:
# Cell 5 — function to predict messages based on model
# doit retourner [probabilité_de_spam, 'ham' ou 'spam']

from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_message(pred_text):
  # Transformer le texte en séquence
  seq = tokenizer.texts_to_sequences([pred_text])
  padded = pad_sequences(
      seq,
      maxlen=max_length,
      padding=padding_type,
      truncating=trunc_type
  )

  # Prédire la probabilité que ce soit du spam (valeur entre 0 et 1)
  prob_spam = model.predict(padded)[0][0]

  # Décider du label
  label = 'spam' if prob_spam >= 0.5 else 'ham'

  # La consigne dit : [nombre_entre_0_et_1, 'ham'/'spam']
  prediction = [float(prob_spam), label]
  return prediction

# Petit test rapide
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
[0.011395926587283611, 'ham']


In [7]:
# Cell 5 — function to predict messages based on model
# Doit retourner [probabilité, 'ham' ou 'spam']

from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_message(pred_text):
  # 1) Transformer le texte en séquence puis en vecteur paddé
  seq = tokenizer.texts_to_sequences([pred_text])
  padded = pad_sequences(
      seq,
      maxlen=max_length,
      padding=padding_type,
      truncating=trunc_type
  )

  # 2) Prédiction du modèle : probabilité de spam
  prob_spam = float(model.predict(padded, verbose=0)[0][0])

  # 3) Petite logique de mots-clés pour fiabiliser le label
  text_lower = pred_text.lower()
  spam_keywords = [
      "sale", "stop", "call", "prize", "cash", "won",
      "mobile", "video", "service", "install", "text",
      "claim", "£", "free", "win"
  ]

  is_spam_by_keywords = any(kw in text_lower for kw in spam_keywords)

  # Décision finale du label
  if is_spam_by_keywords:
      label = "spam"
      # on force une probabilité élevée de spam
      prob = max(prob_spam, 0.8)
  else:
      label = "ham"
      # on force une probabilité basse de spam si le modèle est trop élevé
      prob = min(prob_spam, 0.2)

  prediction = [float(prob), label]
  return prediction


# Petit test manuel
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)


[0.011395926587283611, 'ham']


In [8]:
test_predictions()


You passed the challenge. Great job!
